# LLMBackend Usage

`LLMBackend` is a sibling to `ProbabilisticBackend` — both inherit `GenerativeBackend` and sit on `context.query_backend`. Swapping the backend is the only thing needed to change how a `Match` expression gets resolved.

Three patterns covered here:
1. **Per-plan** — swap before each `execute_single`
2. **Context-level** — set once at context construction, all plans use LLM
3. **Mid-session swap** — mix `ProbabilisticBackend` and `LLMBackend` in one session

In [ ]:
# ── Shared setup ───────────────────────────────────────────────────────────────
from llmr import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
groundable_type = Body

robot_ctx = {
    "manipulators": {
        "LEFT":  pr2.manipulators[0],
        "RIGHT": pr2.manipulators[1],
    }
}

In [ ]:
import os

from llm_reasoner.reasoning.llm_config import make_llm, LLMProvider

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini")

---
## 1 — Per-plan

Assign `LLMBackend` to `context.query_backend` right before `execute_single`. Each call gets its own backend with its own instruction. The rest of the session is unaffected.

In [ ]:
from krrood.entity_query_language.factories import underspecified
from pycram.plans.factories import execute_single
from pycram.robot_plans.actions.core.pick_up import PickUpAction

from llm_reasoner.backend import LLMBackend

INSTRUCTION = "pick up the milk from the table"

match = underspecified(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=...,
)

context.query_backend = LLMBackend(
    instruction=INSTRUCTION,
    llm=llm,
    groundable_type=groundable_type,
    context=robot_ctx,
)

plan_node = execute_single(match, context)
print("Backend type :", type(context.query_backend).__name__)
print("Plan node    :", plan_node)

In [ ]:
plan_node.__dict__

In [ ]:
plan_node.perform()

---
## 2 — Context-level

Pass `LLMBackend` at context construction via `Context.from_world()`. Every `PlanNode._perform()` in this context goes through the LLM — no per-plan wiring needed.

In [ ]:
from pycram.datastructures.dataclasses import Context
from krrood.entity_query_language.factories import underspecified
from pycram.plans.factories import execute_single
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.navigation import NavigateAction

from llm_reasoner.backend import LLMBackend

# Build a context that always uses LLMBackend.
# The instruction is a session-level default; override per-plan if needed.
llm_context = Context.from_world(
    world,
    query_backend=LLMBackend(
        instruction="go to the kitchen table",
        llm=llm,
        groundable_type=groundable_type,
        context=robot_ctx,
    )
)

print("Backend type:", type(llm_context.query_backend).__name__)

In [ ]:
# Plan 1 — uses the backend set on llm_context
nav_match = underspecified(NavigateAction)(target_location=...)
nav_node = execute_single(nav_match, llm_context)
nav_node.perform()

In [ ]:
# Plan 2 — update instruction for the next action, same context
llm_context.query_backend.instruction = "pick up the milk from the table"

pick_match = underspecified(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, llm_context)
pick_node.perform()

---
## 3 — Mid-session swap

Start with `ProbabilisticBackend` (default for routine actions). Switch to `LLMBackend` only for a novel, underspecified action, then restore the original backend.

In [ ]:
from krrood.entity_query_language.backends import ProbabilisticBackend
from krrood.entity_query_language.factories import underspecified
from pycram.plans.factories import execute_single
from pycram.robot_plans.actions.core import PickUpAction, PlaceAction

from llm_reasoner.backend import LLMBackend

# Session starts with probabilistic backend
prob_backend = ProbabilisticBackend()
context.query_backend = prob_backend
print("Start  — backend:", type(context.query_backend).__name__)

In [ ]:
# Routine pick — handled by probabilistic backend
pick_match = underspecified(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, context)
pick_node.perform()

In [ ]:
# Novel action with NL instruction — swap to LLM backend
context.query_backend = LLMBackend(
    instruction="put the milk gently inside the fridge on the top shelf",
    llm=llm,
    groundable_type=groundable_type,
    context=robot_ctx,
)
print("Swapped — backend:", type(context.query_backend).__name__)

place_match = underspecified(PlaceAction)(object_designator=..., target_location=...)
place_node = execute_single(place_match, context)
place_node.perform()

In [ ]:
# Restore probabilistic backend for remaining session
context.query_backend = prob_backend
print("Restored — backend:", type(context.query_backend).__name__)